# 02 · The store protocol (this PC → S3)

**Run first:**
```bash
uv run pytest -v tests/unit/dtrack_left/backends/test_store.py tests/unit/dtrack_right/backends/test_store.py
just test moto
DTRACK_LIVE_KEEP=1 just live test dtrack_left && aws s3 ls --recursive s3://dve-tools/dtrack-live/
```
**Read:** `src/dtrack_left/backends/store.py` (`put_json`, `list_keys`, `commit_unit`) and
`tests/support.py::assert_store_contract`, the contract every store implements.

In [ ]:
import os

import nbkit
import pandas as pd
from nbkit import show, tree

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1 · A unit is committed as payload first, pointer last

The Right side discovers work only through `_INDEX/`. If the pointer appeared first, the Right
side could read a payload that does not exist yet. With this order a crash leaves at worst an
orphan payload that nobody references, never a half-published unit.

In [ ]:
from dtrack_right.backends.store import VolumeStore

folder = nbkit.sandbox("02")
store = VolumeStore(folder / "run")
pointer = store.commit_unit(
    1,
    "orders--2026-01-01--amount--a1b2c3",
    {"stats": "..."},
    {"pair": "orders", "window": "2026-01-01", "column": "AMOUNT"},
)
tree(store.root)
show(pointer)

In [ ]:
store.put_json(
    "units/orders--2026-01-01--status--d4e5f6.json",
    {"stats": "orphan: the process died before the pointer"},
)
print("pointers the Right side sees:", [key for key, _ in store.list_json("_INDEX/")])

## 2 · Zero padding makes lexical order equal commit order

S3 lists keys in lexical order. With `%08d` the Right side can keep the last key it read as a
cursor and ask for `StartAfter=cursor`: it gets exactly the new pointers, in commit order.

In [ ]:
print("unpadded:", sorted(f"{n}-unit" for n in (1, 2, 10, 11)))
print("padded:  ", sorted(f"{n:08d}-unit" for n in (1, 2, 10, 11)))
for n in range(2, 5):
    store.commit_unit(n, f"u{n}", {}, {"pair": "orders", "window": "w", "column": f"C{n}"})
cursor = "_INDEX/00000002-u2.json"
print("after cursor:", store.list_keys("_INDEX/", start_after=cursor))

## 3 · The folder-marker bug that only live tests could catch

When Databricks creates a directory inside an S3-backed Volume, it writes a **0-byte object
whose key ends in `/`** (`manifest/`). `list_json("manifest/")` then tried to parse it as JSON.

**Common mistake:** "moto has a bug". Moto returns exactly what is in the bucket. The unit and
local tests never contained such an object because nothing in them runs Databricks' `mkdir`;
it is an *environment difference*. The fix skips keys ending in `/` in `list_keys`, and the unit
test recreates the marker with a raw `put_object`, exactly as below.

In [ ]:
import boto3
from moto import mock_aws

from dtrack_left.backends.store import S3Store

os.environ |= {
    "AWS_ACCESS_KEY_ID": "x",
    "AWS_SECRET_ACCESS_KEY": "x",
    "AWS_DEFAULT_REGION": "us-east-1",
}
os.environ.pop("DTRACK_S3_ENDPOINT_URL", None)
with mock_aws():
    s3 = boto3.client("s3")
    s3.create_bucket(Bucket="demo")
    s3.put_object(Bucket="demo", Key="run/manifest/", Body=b"")
    left = S3Store("s3://demo/run")
    left.put_json("manifest/orders.json", {"pair": "orders"})
    print("raw listing:", [o["Key"] for o in s3.list_objects_v2(Bucket="demo")["Contents"]])
    print("store view: ", left.list_json("manifest/"))

## Check yourself

<details><summary>What does a KEEP run of test_oracle_to_s3 leave on S3?</summary>

`_STATUS.json` (state `left_row_check_closed`), `_HEARTBEAT`, `row_check/_LEFT_CLOSED`,
`row_check/orders.left.json`, `row_check/customers.left.json`. No `manifest/` or `units/`:
those need the Right side, which that test never starts.
</details>